In [1]:
print("Configuration probe completed.")

Configuration probe completed.


In [29]:
# Repo root => makes the relative "data/..." paths work no matter where Jupyter launched.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=True)

print("repo root:", ROOT)

repo root: /home/dipak/agentic/step9_llmops


In [3]:
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "step9_llmops")

In [19]:
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "NA")
os.environ["GEMINI_MODEL"] = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")

In [30]:
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "NA")
os.environ["GROQ_MODEL"] = os.environ.get("GROQ_MODEL", "openai/gpt-oss-120b")

In [9]:
import warnings
warnings.filterwarnings("ignore", message="Direct use of automatic function calling")

In [5]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
import json
from langchain_core.messages import HumanMessage


In [6]:
USER_PROMPT = "In one sentence, what is RAG?"

In [20]:
gemini_llm = ChatGoogleGenerativeAI(
    model=os.environ["GEMINI_MODEL"],
    google_api_key=os.environ.get("GEMINI_API_KEY"),
)

gemini_response = gemini_llm.invoke([HumanMessage(content=USER_PROMPT)])


In [21]:
if isinstance(gemini_response.content, str):
    gemini_text = gemini_response.content.strip()
else:
    # Newer Gemini models return a list of content blocks
    gemini_text = "".join(
        block.get("text", "")
        for block in gemini_response.content
        if isinstance(block, dict) and block.get("type") == "text"
    ).strip()

print("\n=== Gemini ===")
print(gemini_text)


=== Gemini ===
**Retrieval-Augmented Generation (RAG)** is an AI framework that improves the accuracy and reliability of large language models by fetching relevant information from an external knowledge base to ground the model's generated responses.


In [31]:
groq_llm = ChatGroq(
    model=os.environ["GROQ_MODEL"],
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"],
)

groq_response = groq_llm.invoke([HumanMessage(content=USER_PROMPT)])
groq_text = groq_response.content.strip()

print("=== Groq===")
print(groq_text)

=== Groq===
Retrieval‑Augmented Generation (RAG) is a hybrid AI approach that combines a language model’s generative capabilities with a real‑time search of external documents or databases, retrieving relevant information to ground and improve the model’s output.


In [33]:
from openai import OpenAI

In [34]:
# ---------- 3. Judge — OpenAI-compatible endpoint ----------
judge_client = OpenAI(
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
)

In [37]:
JUDGE_PROMPT = f"""You are an evaluation judge. Question: "{USER_PROMPT}"

Answer A (Groq): {groq_text}
Answer B (Gemini): {gemini_text}

Return ONLY valid JSON, no markdown fences, no prose, exactly this schema:
{{
  "winner": "A" | "B" | "tie",
  "reasoning": "<one sentence>",
  "score_a": <int 1-10>,
  "score_b": <int 1-10>
}}"""


In [38]:
judge_response = judge_client.chat.completions.create(
    model=os.environ["LLM_MODEL"],
    messages=[{"role": "user", "content": JUDGE_PROMPT}],
    temperature=0,
)

In [39]:
judge_response

ChatCompletion(id='chatcmpl-97ecefa902674f36', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "winner": "B",\n  "reasoning": "More concise and directly addresses the question.",\n  "score_a": 7,\n  "score_b": 9\n}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning=None), stop_reason=None, token_ids=None)], created=1789075033, model='Qwen/Qwen2.5-7B-Instruct-AWQ', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=40, prompt_tokens=217, total_tokens=257, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, kv_transfer_params=None)

In [40]:
judge_raw = judge_response.choices[0].message.content.strip()

In [41]:
# defensive: some models wrap JSON in ```json ... ``` anyway
if judge_raw.startswith("```"):
    judge_raw = judge_raw.strip("`").removeprefix("json").strip()

judge_json = json.loads(judge_raw)

In [42]:
judge_json

{'winner': 'B',
 'reasoning': 'More concise and directly addresses the question.',
 'score_a': 7,
 'score_b': 9}

In [43]:
expected_keys = {"winner", "reasoning", "score_a", "score_b"}
missing = expected_keys - judge_json.keys()
assert not missing, f"Missing keys in judge output: {missing}"
assert judge_json["winner"] in ("A", "B", "tie"), f"Bad winner: {judge_json['winner']}"
assert isinstance(judge_json["score_a"], int) and 1 <= judge_json["score_a"] <= 10
assert isinstance(judge_json["score_b"], int) and 1 <= judge_json["score_b"] <= 10

print("\n=== Judge ===")
print(json.dumps(judge_json, indent=2))


=== Judge ===
{
  "winner": "B",
  "reasoning": "More concise and directly addresses the question.",
  "score_a": 7,
  "score_b": 9
}
